In [75]:
import duckdb
import pandas as pd

In [76]:
conn = duckdb.connect("lego.db", read_only=True)

In [77]:
#verify tables and number of rows

tables = conn.sql("SHOW TABLES").fetchall() #sql command native to duckdb, to retrieve all table names


#print row count per table in an ASCII table 
print(f"{'Table Name':<25} | {'Total Rows':>10}")
print("-" * 40)

for table in tables:
    t_name = table[0]
    count = conn.sql(f"SELECT COUNT(*) FROM {t_name}").fetchone()[0]
    print(f"{t_name:<25} | {count:>10,}")

Table Name                | Total Rows
----------------------------------------
colors                    |        275
elements                  |    110,700
inventories               |     45,518
inventory_minifigs        |     25,142
inventory_parts           |  1,497,894
inventory_sets            |      4,983
minifigs                  |     16,740
part_categories           |         76
part_relationships        |     36,130
parts                     |     62,460
sets                      |     26,870
themes                    |        494


In [78]:

#store dataframes of each table as a dictionary
dfs = {}

for table in tables:
    t = table[0]
    dfs[t] = conn.sql(f"SELECT * FROM {t}").df()

In [79]:
pd.set_option("display.width", 1000)

In [80]:
#table memory usage before cleaning and typecasting
total_bytes = sum(df.memory_usage(deep=True).sum() for df in dfs.values())
total_mb = total_bytes / (1024 ** 2)
print(f"Combined Memory Usage: {total_mb:.2f} MB")

Combined Memory Usage: 313.55 MB


In [81]:
#table info before cleaning and typecasting
for name, df in dfs.items():
    print(f"rows in '{name}': {len(df)}")
    print(df.info())
    print()
    print("------------------------------------------------------------------------------------------------------------------------")
    print()

rows in 'colors': 275
<class 'pandas.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         275 non-null    int64
 1   name       275 non-null    str  
 2   rgb        275 non-null    str  
 3   is_trans   275 non-null    bool 
 4   num_parts  275 non-null    int64
 5   num_sets   275 non-null    int64
 6   y1         273 non-null    Int64
 7   y2         273 non-null    Int64
dtypes: Int64(2), bool(1), int64(3), str(2)
memory usage: 16.0 KB
None

------------------------------------------------------------------------------------------------------------------------

rows in 'elements': 110700
<class 'pandas.DataFrame'>
RangeIndex: 110700 entries, 0 to 110699
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   element_id  110700 non-null  int64
 1   part_num    110700 non-null  str  
 2   color_id    110700 n

In [82]:
for name, df in dfs.items():
    for column_name, column in df.items():
        #tables with null values will use Pandas integers (Int..)
        if column.dtype == 'Int64':
            df[column_name] = df[column_name].astype('Int32')
        #tables without null values will use NumPy integers (int..)
        if column.dtype == 'int64':
            df[column_name] = df[column_name].astype('int32')

In [83]:
#table memory usage after cleaning and typecasting
total_bytes = sum(df.memory_usage(deep=True).sum() for df in dfs.values())
total_mb = total_bytes / (1024 ** 2)
print(f"Combined Memory Usage: {total_mb:.2f} MB")

Combined Memory Usage: 293.94 MB


In [84]:
#table info after cleaning and typecasting
for name, df in dfs.items():
    print(f"rows in '{name}': {len(df)}")
    print(df.info())
    print()
    print("------------------------------------------------------------------------------------------------------------------------")
    print()

rows in 'colors': 275
<class 'pandas.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         275 non-null    int32
 1   name       275 non-null    str  
 2   rgb        275 non-null    str  
 3   is_trans   275 non-null    bool 
 4   num_parts  275 non-null    int32
 5   num_sets   275 non-null    int32
 6   y1         273 non-null    Int32
 7   y2         273 non-null    Int32
dtypes: Int32(2), bool(1), int32(3), str(2)
memory usage: 10.6 KB
None

------------------------------------------------------------------------------------------------------------------------

rows in 'elements': 110700
<class 'pandas.DataFrame'>
RangeIndex: 110700 entries, 0 to 110699
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   element_id  110700 non-null  int32
 1   part_num    110700 non-null  str  
 2   color_id    110700 n

In [85]:
#table heads
for name, df in dfs.items():
    print(name)
    print(df.head(10))
    print()
    print("------------------------------------------------------------------------------------------------------------------------")
    print()


colors
   id            name     rgb  is_trans  num_parts  num_sets    y1    y2
0  -1       [Unknown]  0033B2     False         17         2  2000  2000
1   0           Black  05131D     False     809671    224869  1957  2026
2   1            Blue  0055BF     False     203471     49321  1949  2026
3   2           Green  237841     False      88963     26311  1949  2026
4   3  Dark Turquoise  008F9B     False      23041      6114  1998  2026
5   4             Red  C91A09     False     313462     93838  1949  2026
6   5       Dark Pink  C870A0     False      15416      4788  2003  2026
7   6           Brown  583927     False       9551      3786  1974  2006
8   7      Light Gray  9BA19D     False      92813     25173  1954  2007
9   8       Dark Gray  6D6E5C     False      22442      7797  1978  2006

------------------------------------------------------------------------------------------------------------------------

elements
   element_id         part_num  color_id  design_id
0     